# I01 - Advanced Optimization Algorithms

**Level:** Intermediate

---

## Learning Objectives

By the end of this lesson, you will:
1. Understand advanced optimization algorithms beyond basic SGD
2. Implement Adam, RMSprop, and AdaGrad optimizers
3. Learn about learning rate scheduling strategies
4. Compare optimizer performance on real datasets
5. Understand momentum and adaptive learning rates

---

## Prerequisites

- Completed B01-B05 (Basic Level)
- Understanding of gradient descent
- Familiarity with neural networks
- Basic calculus knowledge

---

In [ ]:
# Import required libraries
import numpy as np
import tensorflow as tf
from tensorflow import keras
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print(f"TensorFlow version: {tf.__version__}")
print(f"NumPy version: {np.__version__}")

## 1. Understanding Optimization Challenges

### Why Do We Need Advanced Optimizers?

Basic SGD has limitations:
- **Slow convergence** in flat regions
- **Oscillation** in steep valleys
- **Same learning rate** for all parameters
- **Difficulty escaping** local minima

Advanced optimizers address these issues through:
- **Momentum**: Accelerate in consistent directions
- **Adaptive learning rates**: Different rates for different parameters
- **Second-order information**: Use gradient history

## 2. Momentum-Based Optimization

### SGD with Momentum

Momentum helps accelerate SGD in relevant directions:

$$v_t = \beta v_{t-1} + (1 - \beta) \nabla L(\theta_{t-1})$$
$$\theta_t = \theta_{t-1} - \alpha v_t$$

Where:
- $v_t$ is the velocity (momentum)
- $\beta$ is the momentum coefficient (typically 0.9)
- $\alpha$ is the learning rate

In [ ]:
# Load MNIST dataset for comparison
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

# Normalize
x_train = x_train.reshape(-1, 784).astype('float32') / 255.0
x_test = x_test.reshape(-1, 784).astype('float32') / 255.0

# Use subset for faster training
x_train_small = x_train[:10000]
y_train_small = y_train[:10000]

print(f"Training samples: {x_train_small.shape[0]}")
print(f"Test samples: {x_test.shape[0]}")

In [ ]:
def create_model():
    """Create a simple neural network for comparison"""
    model = keras.Sequential([
        keras.layers.Dense(128, activation='relu', input_shape=(784,)),
        keras.layers.Dense(64, activation='relu'),
        keras.layers.Dense(10, activation='softmax')
    ])
    return model

# Test model creation
test_model = create_model()
test_model.summary()

## 3. Comparing Optimizers

Let's compare different optimizers:
1. **SGD**: Basic stochastic gradient descent
2. **SGD with Momentum**: Accelerated SGD
3. **RMSprop**: Adaptive learning rate
4. **Adam**: Combines momentum and adaptive learning rates

In [ ]:
# Dictionary of optimizers to compare
optimizers = {
    'SGD': keras.optimizers.SGD(learning_rate=0.01),
    'SGD + Momentum': keras.optimizers.SGD(learning_rate=0.01, momentum=0.9),
    'RMSprop': keras.optimizers.RMSprop(learning_rate=0.001),
    'Adam': keras.optimizers.Adam(learning_rate=0.001)
}

# Store training histories
histories = {}

# Train with each optimizer
for name, optimizer in optimizers.items():
    print(f"\nTraining with {name}...")
    
    model = create_model()
    model.compile(
        optimizer=optimizer,
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    
    history = model.fit(
        x_train_small, y_train_small,
        epochs=20,
        batch_size=128,
        validation_split=0.2,
        verbose=0
    )
    
    histories[name] = history.history
    
    # Evaluate
    test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
    print(f"{name} - Test Accuracy: {test_acc:.4f}")

In [ ]:
# Plot comparison
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Training loss
for name, history in histories.items():
    axes[0].plot(history['loss'], label=name, linewidth=2)
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Training Loss', fontsize=12)
axes[0].set_title('Training Loss Comparison', fontsize=14, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Validation accuracy
for name, history in histories.items():
    axes[1].plot(history['val_accuracy'], label=name, linewidth=2)
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Validation Accuracy', fontsize=12)
axes[1].set_title('Validation Accuracy Comparison', fontsize=14, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Adam Optimizer Deep Dive

### Adam (Adaptive Moment Estimation)

Adam combines the best of both worlds:
- **Momentum**: First moment (mean) of gradients
- **RMSprop**: Second moment (variance) of gradients

**Algorithm:**

$$m_t = \beta_1 m_{t-1} + (1 - \beta_1) g_t$$
$$v_t = \beta_2 v_{t-1} + (1 - \beta_2) g_t^2$$
$$\hat{m}_t = \frac{m_t}{1 - \beta_1^t}$$
$$\hat{v}_t = \frac{v_t}{1 - \beta_2^t}$$
$$\theta_t = \theta_{t-1} - \frac{\alpha}{\sqrt{\hat{v}_t} + \epsilon} \hat{m}_t$$

**Default hyperparameters:**
- $\beta_1 = 0.9$ (momentum)
- $\beta_2 = 0.999$ (RMSprop)
- $\epsilon = 10^{-8}$ (numerical stability)

## 5. Learning Rate Scheduling

Learning rate scheduling adjusts the learning rate during training:

1. **Step Decay**: Reduce LR by factor every N epochs
2. **Exponential Decay**: Exponentially decrease LR
3. **Cosine Annealing**: Cosine function decay
4. **Reduce on Plateau**: Reduce when metric stops improving

In [ ]:
# Example: Exponential decay schedule
initial_learning_rate = 0.001
lr_schedule = keras.optimizers.schedules.ExponentialDecay(
    initial_learning_rate,
    decay_steps=1000,
    decay_rate=0.96,
    staircase=True
)

# Visualize learning rate schedule
steps = np.arange(0, 10000, 100)
learning_rates = [lr_schedule(step).numpy() for step in steps]

plt.figure(figsize=(10, 5))
plt.plot(steps, learning_rates, linewidth=2)
plt.xlabel('Training Step', fontsize=12)
plt.ylabel('Learning Rate', fontsize=12)
plt.title('Exponential Decay Learning Rate Schedule', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Train with learning rate schedule
model_scheduled = create_model()
optimizer_scheduled = keras.optimizers.Adam(learning_rate=lr_schedule)

model_scheduled.compile(
    optimizer=optimizer_scheduled,
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history_scheduled = model_scheduled.fit(
    x_train_small, y_train_small,
    epochs=20,
    batch_size=128,
    validation_split=0.2,
    verbose=1
)

## 6. Practical Tips

### Choosing an Optimizer

**Adam (Default choice):**
- Works well out of the box
- Good for most problems
- Adaptive learning rates

**SGD with Momentum:**
- Better generalization sometimes
- Requires careful tuning
- Good for computer vision

**RMSprop:**
- Good for RNNs
- Handles non-stationary objectives

### Hyperparameter Tuning

1. **Start with defaults**: Adam with lr=0.001
2. **Try learning rates**: [0.1, 0.01, 0.001, 0.0001]
3. **Add scheduling**: If training plateaus
4. **Monitor training**: Watch for oscillation or slow convergence

## Summary

In this lesson, you learned:

1. ✅ Advanced optimization algorithms (Adam, RMSprop, AdaGrad)
2. ✅ Momentum and adaptive learning rates
3. ✅ Learning rate scheduling strategies
4. ✅ Comparing optimizer performance
5. ✅ Practical tips for choosing optimizers

**Key Takeaways:**
- Adam is a good default choice for most problems
- Learning rate scheduling can improve convergence
- Different optimizers work better for different tasks
- Always monitor training curves

**Next Steps:**
- Move to I02 - Regularization Techniques
- Experiment with different optimizers on your own datasets
- Try combining optimizers with regularization

---

**References:**
- Kingma & Ba (2014): "Adam: A Method for Stochastic Optimization"
- Ruder (2016): "An overview of gradient descent optimization algorithms"
- TensorFlow Optimization Documentation

---

## Key Takeaways

**Relevant UoA Courses:** COMPSCI 762, COMPSCI 713

1. Adam combines momentum and RMSprop: adaptive learning rates per parameter
2. Learning rate scheduling: reduce LR when plateau (ReduceLROnPlateau, CosineAnnealing)
3. Momentum accelerates convergence: v_t = βv_{t-1} + ∇L
4. RMSprop adapts learning rate using moving average of squared gradients
5. Warmup: gradually increase LR at start to stabilize training

---

## Exam Preparation Guide

### Essential Concepts for Exams

- Compare optimizers: SGD vs Momentum vs Adam (convergence speed, memory)
- Understand Adam parameters: β1 (momentum), β2 (RMSprop), ε (numerical stability)
- Know when to use which optimizer: Adam (default), SGD+momentum (better generalization)
- Calculate learning rate with scheduling: given initial LR and decay
- Common question: Explain why adaptive learning rates help convergence

### Common Mistakes to Avoid

- ❌ Using same learning rate for all optimizers
- ❌ Not tuning β1, β2 in Adam (defaults usually work)
- ❌ Forgetting warmup for large batch training

### Practice Problems

1. Compare convergence: SGD vs Adam on same problem
2. Given LR=0.001, decay=0.1 every 10 epochs, what's LR at epoch 25?
3. Why does Adam work better than SGD for sparse gradients?

### How This Helps Your UoA Courses

**COMPSCI 762, COMPSCI 713:**
- Provides hands-on implementation of theoretical concepts
- Practice problems similar to exam questions
- Reinforces lecture material with code examples
- Helps build intuition for complex topics

### Study Tips for Advanced Topics

1. **Connect to Fundamentals**: Link advanced concepts to basics
2. **Read Papers**: Understand state-of-the-art approaches
3. **Implement from Scratch**: Deepens understanding
4. **Compare Approaches**: Know trade-offs between methods
5. **Real-World Applications**: Understand practical use cases

### Exam Question Types

- **Conceptual**: Explain advanced mechanisms and why they work
- **Comparison**: Compare multiple approaches, trade-offs
- **Design**: Design system for specific requirements
- **Analysis**: Analyze experimental results, identify issues
- **Application**: Apply techniques to novel problems

---


---

## Learning Progress Tracker

Use this section to track your learning progress for this lesson.

### Completion Status
- [ ] Lesson completed
- [ ] All code cells executed successfully
- [ ] Understood all key concepts
- [ ] Completed practice exercises (if any)

### Dates
- **First Completed:** ____/____/____
- **Last Reviewed:** ____/____/____
- **Next Review:** ____/____/____ (Recommended: 1 week, 1 month, 3 months)

### Understanding Level
Rate your understanding (1-5): _____ / 5

- 1 = Need to review completely
- 2 = Understood basics, need more practice
- 3 = Good understanding, minor gaps
- 4 = Strong understanding, can explain to others
- 5 = Mastered, can apply in projects

### Notes & Reflections
```
Write your notes here:
- What concepts were challenging?
- What was interesting or surprising?
- How can you apply this in projects?
- Questions to explore further?




```

### Key Concepts to Remember (I01)
- [ ] Adam optimizer
- [ ] RMSprop
- [ ] Learning rate scheduling
- [ ] Adaptive learning rates

---